In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from tabulate import tabulate
import matplotlib.pyplot as plt

import geopandas as gpd

# Investigating DLL data on urban areas

In [3]:
gpkg_path = Path("../data/input/DLL") / "data_final.gpkg"

current_dir = Path.cwd()
print(f"Current working directory: {current_dir}")

# List layers - requires pyogrio (the current geopandas default)
layers_info = gpd.list_layers(gpkg_path)
print("\nLayers in file:")
print(layers_info)

layer_name = layers_info["name"].iloc[0]
gdf = gpd.read_file(gpkg_path, layer=layer_name)

print("\nNumber of features:", len(gdf))
print("Columns:", gdf.columns.tolist())
print("Geometry type(s):", gdf.geom_type.unique())
print("CRS:", gdf.crs)
print("Bounding box:", gdf.total_bounds)
print(gdf.head())

Current working directory: k:\PythonWork\downscaling\Kaya_downscaling\downscaling

Layers in file:
         name geometry_type
0  data_final  MultiPolygon

Number of features: 356508
Columns: ['UID', 'NAME_1', 'NAME_2', 'NAME_3', 'NAME_4', 'NAME_5', 'GGMCF', 'EDGAR', 'DEGURBA_L1', 'DEGURBA_L2', 'Growth_Rate', 'GGMCF_2022', 'GID_0', 'NAME_0', 'GID_1', 'ENGTYPE_1', 'GID_2', 'ENGTYPE_2', 'GID_3', 'ENGTYPE_3', 'GID_4', 'ENGTYPE_4', 'GID_5', 'ENGTYPE_5', 'CONTINENT', 'GDAM_ID', 'POP', 'GDP', 'BUILT_SUM', 'BUILT_SUM_BASE', 'imp_change_area', 'imp_total_base', 'imp_total', 'ELEC_SUM', 'area', 'GDP_PC', 'POP_DENS', 'imp_prop', 'imp_prop_base', 'built_prop', 'built_prop_base', 'built_prop_absolute_trend', 'built_prop_relative_trend', 'built_prop_relative_trend_cut', 'country_area_sum', 'country_gdp_sum', 'country_pop_sum', 'country_built_sum', 'region_area_sum', 'region_gdp_sum', 'region_pop_sum', 'region_built_sum', 'area_prop_adm_0', 'GDP_prop_adm_0', 'POP_prop_adm_0', 'BUILT_prop_adm_0', 'ar

In [4]:
# print first row
GID_0s = ["NLD"]
selected = gdf[gdf["GID_0"].isin(GID_0s)].drop(columns="geometry")
print(tabulate(selected, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))

+---------+----------------------+------------------------+----------+----------+----------+-----------------+------------------+--------------+--------------+---------------+-----------------+---------+-------------+----------+-------------+--------------+--------------+---------+-------------+---------+-------------+---------+-------------+-------------+--------------+-----------+------------------+--------------+------------------+-------------------+------------------+---------------+-----------------+-----------------+-------------+------------+------------+-----------------+--------------+-------------------+-----------------------------+-----------------------------+---------------------------------+--------------------+-------------------+-------------------+---------------------+-------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+--------------------+-------------------+------------------+---

In [13]:
# read in csv file with time series data
csv_path = Path("../data/input/DLL") / "data_timeseries_70.csv"
df = pd.read_csv(csv_path, sep=",")
df

,GDAM_id,GID_1,cluster_2015,cluster_2020,cluster_2030,cluster_2040,cluster_2050,cluster_2060,cluster_2070,cluster_2080,cluster_2090,cluster_2100
0,AFG.1.1_1,AFG.1_1,0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
1,AFG.1.2_1,AFG.1_1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AFG.1.3_1,AFG.1_1,0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,AFG.1.4_1,AFG.1_1,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,AFG.1.5_1,AFG.1_1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
356456,ZWE.10.13.15_2,ZWE.10_1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
356457,ZWE.10.13.16_2,ZWE.10_1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
356458,ZWE.10.13.17_2,ZWE.10_1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
356459,ZWE.10.13.18_2,ZWE.10_1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# Combine geopandas dataframe with csv dataframe on GDAM_ID

gpkg_path = Path("../data/input/DLL") / "data_final.gpkg"
csv_path = Path("../data/input/DLL") / "data_timeseries_70.csv"
merged_output_path = Path("../data/processed/DLL") / "urban_classification_years.parquet"

# Only load the ID column plus geometry from the gpkg first, to check the join before pulling everything in
gdf = gpd.read_file(gpkg_path, layer="data_final")
df_ts = pd.read_csv(csv_path)

# Check join coverage before committing to a full merge
gpkg_ids = set(gdf["GDAM_ID"])
csv_ids = set(df_ts["GDAM_id"])

print("IDs in gpkg but not in csv:", len(gpkg_ids - csv_ids))
print("IDs in csv but not in gpkg:", len(csv_ids - gpkg_ids))

merged_gdf = gdf.merge(df_ts, left_on="GDAM_ID", right_on="GDAM_id", how="left")
merged_gdf.to_parquet(merged_output_path)

IDs in gpkg but not in csv: 28
IDs in csv but not in gpkg: 0


In [16]:
# check
missing_ids_from_csv = gpkg_ids - csv_ids
print("IDs in gpkg but not in csv:", missing_ids_from_csv)
print("-----------------------------------------------------")
missing_rows = gdf[gdf["GDAM_ID"].isin(missing_ids_from_csv)]
print(missing_rows[["GDAM_ID", "NAME_0", "NAME_1", "NAME_2", "CONTINENT"]])

IDs in gpkg but not in csv: {'GBR.1.6.1.3_1', 'MCO', 'GBR.1.6.1.14_1', 'MAF', 'CXR', 'CCK', 'ATA', 'GBR.3.27.1.3_1', 'FLK', 'GIB', 'CUW', 'GBR.1.6.1.20_1', 'SGS', 'PCN', 'VAT', 'XPI', 'BVT', 'XCA', 'NIU', 'KIR', 'MDV', 'SXM', 'XSP', 'HMD', 'IOT', 'XCL', 'NFK', 'ABW'}
-----------------------------------------------------
               GDAM_ID                            NAME_0    NAME_1  \
2856               ATA                        Antarctica      None   
3378               ABW                             Aruba      None   
19094              BVT                     Bouvet Island      None   
24667              IOT    British Indian Ocean Territory      None   
41684              XCA                       Caspian Sea      None   
44870              CXR                  Christmas Island      None   
44871              XCL                 Clipperton Island      None   
44872              CCK                     Cocos Islands      None   
47412              CUW                          

In [17]:
# print info merged data
print("Shape (rows, columns):", merged_gdf.shape)
print("Columns:", merged_gdf.columns.tolist())
print("Dtypes:")
print(merged_gdf.dtypes)
print("CRS:", merged_gdf.crs)
print("Geometry column:", merged_gdf.geometry.name)
print("Geometry type(s):", merged_gdf.geom_type.unique())
print("Memory usage (bytes, deep):")
print(merged_gdf.memory_usage(deep=True))
print("Total memory usage (MB):", merged_gdf.memory_usage(deep=True).sum() / 1e6)

print(merged_gdf.describe())
merged_gdf.info(memory_usage="deep")

Shape (rows, columns): (356508, 80)
Columns: ['UID', 'NAME_1', 'NAME_2', 'NAME_3', 'NAME_4', 'NAME_5', 'GGMCF', 'EDGAR', 'DEGURBA_L1', 'DEGURBA_L2', 'Growth_Rate', 'GGMCF_2022', 'GID_0', 'NAME_0', 'GID_1_x', 'ENGTYPE_1', 'GID_2', 'ENGTYPE_2', 'GID_3', 'ENGTYPE_3', 'GID_4', 'ENGTYPE_4', 'GID_5', 'ENGTYPE_5', 'CONTINENT', 'GDAM_ID', 'POP', 'GDP', 'BUILT_SUM', 'BUILT_SUM_BASE', 'imp_change_area', 'imp_total_base', 'imp_total', 'ELEC_SUM', 'area', 'GDP_PC', 'POP_DENS', 'imp_prop', 'imp_prop_base', 'built_prop', 'built_prop_base', 'built_prop_absolute_trend', 'built_prop_relative_trend', 'built_prop_relative_trend_cut', 'country_area_sum', 'country_gdp_sum', 'country_pop_sum', 'country_built_sum', 'region_area_sum', 'region_gdp_sum', 'region_pop_sum', 'region_built_sum', 'area_prop_adm_0', 'GDP_prop_adm_0', 'POP_prop_adm_0', 'BUILT_prop_adm_0', 'area_prop_adm_1', 'GDP_prop_adm_1', 'POP_prop_adm_1', 'BUILT_prop_adm_1', 'Cluster_DDL', 'GGMCF_PC', 'EDGAR_PC', 'Emi_Gap', 'Tot_Diff', 'NetConsume

k:\PythonWork\downscaling\Kaya_downscaling\.pixi\envs\default\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
k:\PythonWork\downscaling\Kaya_downscaling\.pixi\envs\default\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


                 UID         GGMCF         EDGAR     DEGURBA_L1  \
count  356508.000000  3.564830e+05  3.564830e+05  356484.000000   
mean   178254.500000  9.034151e+07  9.006887e+07       1.613012   
std    102915.139222  8.710752e+08  9.445564e+08       0.691220   
min         1.000000  0.000000e+00  0.000000e+00       1.000000   
25%     89127.750000  5.436145e+05  3.638578e+05       1.000000   
50%    178254.500000  3.233557e+06  2.425764e+06       1.000000   
75%    267381.250000  1.801825e+07  1.187864e+07       2.000000   
max    356508.000000  1.462826e+11  1.006457e+11       3.000000   

          DEGURBA_L2    Growth_Rate    GGMCF_2022           POP           GDP  \
count  356484.000000  336343.000000  3.363420e+05  3.564800e+05  3.564800e+05   
mean       17.769796       0.270965  9.965060e+07  2.080947e+04  3.139627e+08   
std         6.332552       0.754798  9.554070e+08  1.365896e+05  3.364996e+09   
min        11.000000      -0.309783  0.000000e+00  0.000000e+00  0.00000